# F4 — Compare every variant, and decide whether to ship one

The project now has several ways to put a phone-tier image into SigLIP's
space. This notebook scores them on one protocol and makes the deployment
trade explicit, so the decision rests on numbers rather than novelty.

| Variant | Artifact | Fit cost | Refit on teacher upgrade |
|---|---|---|---|
| linear adapter (B2) | 0.79 MB matrix | seconds | seconds |
| MLP control (B2-bis) | ~2 MB | ~2 min | minutes |
| distilled head (F2) | ~5 MB | ~10 min | minutes |
| unfrozen student (F3) | full weights | hours | full retrain |

**The gate for shipping anything new:** it must beat the adapter's R@5 of
0.842 by a margin that survives the eval set's noise, and the operational
cost must be worth that margin. A student that wins by 0.01 and forfeits
seconds-long refits is not obviously an improvement.

In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
import numpy as np, torch, json
from pathlib import Path
DATA_DIR = Path(os.environ["DATA_DIR"])
DEV = "cuda" if torch.cuda.is_available() else "cpu"

pairs = np.load(str(DATA_DIR / "pairs.npz"))
ad = np.load(str(DATA_DIR / "adapter.npz"))
te = ad["eval_idx"]

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
def recall(S):
    o = np.argsort(-S, axis=1)
    r = (o == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

txt = l2n(pairs["sig_txt"][te].astype(np.float64))
sig = l2n(pairs["sig_img"][te].astype(np.float64))
mob = pairs["mob_img"][te].astype(np.float64)

variants = {}
variants["ceiling (SigLIP native)"] = sig
variants["raw (no map)"] = l2n(np.pad(mob, ((0,0),(0, sig.shape[1]-mob.shape[1]))))
variants["linear adapter (B2)"] = l2n(mob @ ad["W_ridge"].astype(np.float64))

f2 = DATA_DIR / "f2_head.pt"
if f2.exists():
    import torch.nn as nn, torch.nn.functional as F
    head = nn.Sequential(nn.Linear(mob.shape[1], 2048), nn.GELU(),
                         nn.Linear(2048, sig.shape[1])).to(DEV)
    sd = torch.load(str(f2), map_location=DEV)
    head.load_state_dict({k.replace("net.", ""): v for k, v in sd.items()})
    head.eval()
    with torch.no_grad():
        variants["distilled head (F2)"] = l2n(
            head(torch.tensor(mob).float().to(DEV)).cpu().numpy())

res = {n: recall(txt @ v.T) for n, v in variants.items()}
c = res["ceiling (SigLIP native)"]
print(f"{'variant':28s} {'R@1':>7s} {'R@5':>7s} {'R@10':>7s}   "
      f"{'% ceiling @5':>13s}")
for n, r in res.items():
    print(f"{n:28s} " + "  ".join(f"{r[k]:.3f}" for k in (1,5,10)) +
          f"   {100*r[5]/c[5]:12.1f}%")

In [ ]:
# is the difference real, or eval-set noise? bootstrap the gap
def boot_gap(a, b, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    Sa, Sb = txt @ a.T, txt @ b.T
    ra = (np.argsort(-Sa, 1) == np.arange(len(Sa))[:, None]).argmax(1)
    rb = (np.argsort(-Sb, 1) == np.arange(len(Sb))[:, None]).argmax(1)
    hit_a, hit_b = (ra < 5).astype(float), (rb < 5).astype(float)
    gaps = [ (hit_a[i].mean() - hit_b[i].mean())
             for i in (rng.integers(0, len(hit_a), len(hit_a))
                       for _ in range(n)) ]
    lo, hi = np.percentile(gaps, [2.5, 97.5])
    return float(np.mean(gaps)), float(lo), float(hi)

base = variants["linear adapter (B2)"]
for n, v in variants.items():
    if n in ("ceiling (SigLIP native)", "raw (no map)",
             "linear adapter (B2)"):
        continue
    m, lo, hi = boot_gap(v, base)
    verdict = ("REAL improvement" if lo > 0 else
               ("REAL regression" if hi < 0 else
                "within noise - not a difference"))
    print(f"{n} vs linear adapter, R@5 gap {m:+.4f} "
          f"[{lo:+.4f}, {hi:+.4f}]  -> {verdict}")

## The deployment decision

Ship a new variant only when three things hold together:

1. the R@5 bootstrap interval excludes zero — the gain is not eval noise;
2. the margin is large enough to matter for the product surface (a results
   grid cares about R@5 and R@10, a single-answer flow cares about R@1);
3. the operational cost is acceptable — a student that gains 0.01 and
   turns a seconds-long refit into an overnight retrain is a regression in
   everything except the score.

This project's stated position stands unless the numbers overturn it: the
0.79 MB matrix is the artifact, because it wins on cost, auditability and
upgrade agility, and loses only where MobileCLIP's own capacity binds.